# Day 29 - Road Damage Detection, trained on a free Colab GPU

**Before running:** `Runtime` -> `Change runtime type` -> Hardware accelerator = **T4 GPU** -> Save.

Then `Runtime` -> `Run all`. You'll be prompted for a Roboflow API key in the cell below (free account at roboflow.com -> Settings -> API Keys).

This trains **three variants** so we have a real, evidence-based comparison instead of guessing:
- `8class` - the original assigned dataset, all 8 damage classes
- `5class` - same dataset, 3 near-empty classes (Ravelling/Rutting/Striping, 13-30 instances each) removed
- `2class` - **pothole vs. crack** - every crack/surface-damage type (including the 3 rare ones - no data thrown away) folded into one `crack` class. Coarser, but a much easier, well-balanced problem (pothole=4,620 vs crack=5,312 instances) - this is the one aimed at clearing 80% mAP50.

At the end, all three get evaluated on the held-out test split and you download the best-scoring `best.pt` to drop into your local `Day29/` folder (plus the others, for comparison).

In [ ]:
!pip install -q ultralytics roboflow
import torch
print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only - go set the runtime type to T4 GPU first!')

In [ ]:
from getpass import getpass
from roboflow import Roboflow

api_key = getpass('Roboflow API key: ')
rf = Roboflow(api_key=api_key)
project = rf.workspace('road-damage-detection-ds22n').project('road-damage-dataset-8jvz5')
version = project.version(2)
dataset = version.download('yolov8', location='dataset', overwrite=True)
print('Downloaded to', dataset.location)

In [ ]:
# Build the two reduced-taxonomy variants. See README.md "Challenges" for the full
# reasoning: Ravelling/Rutting/Striping have 13-30 total instances each - nowhere near
# enough to learn from, and mAP50 is the plain average across classes, so they
# mechanically cap the whole metric no matter how well the other classes are learned.
import shutil
from pathlib import Path
import yaml

SRC = Path('dataset')
ORIGINAL_NAMES = ['Alligator', 'Edge Cracking', 'Lateral-Crack', 'Longitudinal-Crack',
                   'Ravelling', 'Rutting', 'Striping', 'pothole']
RARE = {'Ravelling', 'Rutting', 'Striping'}

def build_variant(dest_name, new_names, remap):
    dest = Path(dest_name)
    for split in ['train', 'valid', 'test']:
        src_img, src_lbl = SRC / split / 'images', SRC / split / 'labels'
        dst_img, dst_lbl = dest / split / 'images', dest / split / 'labels'
        dst_img.mkdir(parents=True, exist_ok=True)
        dst_lbl.mkdir(parents=True, exist_ok=True)
        for img in src_img.iterdir():
            shutil.copy2(img, dst_img / img.name)
        for lbl_path in src_lbl.glob('*.txt'):
            out_lines = []
            for line in lbl_path.read_text().strip().splitlines():
                if not line.strip():
                    continue
                parts = line.split()
                new_id = remap[int(parts[0])]
                if new_id is not None:
                    out_lines.append(' '.join([str(new_id), *parts[1:]]))
            (dst_lbl / lbl_path.name).write_text(('\n'.join(out_lines) + '\n') if out_lines else '')
    yaml.dump({'train': '../train/images', 'val': '../valid/images', 'test': '../test/images',
               'nc': len(new_names), 'names': new_names}, open(dest / 'data.yaml', 'w'), sort_keys=False)
    print(f'Built {dest_name}/ ({len(new_names)} classes)')

# 5class: drop the 3 rare classes entirely
names_5 = ['Alligator', 'Edge Cracking', 'Lateral-Crack', 'Longitudinal-Crack', 'pothole']
remap_5 = {}
for i, name in enumerate(ORIGINAL_NAMES):
    remap_5[i] = None if name in RARE else (4 if name == 'pothole' else i)
build_variant('dataset_dropped5', names_5, remap_5)

# 2class: pothole vs. everything else ("crack") - no data dropped, just coarser
names_2 = ['crack', 'pothole']
remap_2 = {i: (1 if name == 'pothole' else 0) for i, name in enumerate(ORIGINAL_NAMES)}
build_variant('dataset_2class', names_2, remap_2)

In [ ]:
from ultralytics import YOLO

def train_and_eval(data_yaml, name, epochs=60):
    model = YOLO('yolov8n.pt')
    # Use the actual returned save_dir, not a guessed path - Ultralytics' internal
    # project/runs_dir joining can put this a level deeper than project='runs' name=name
    # would suggest (the same nested-path quirk documented in README.md "Challenges").
    train_out = model.train(data=data_yaml, epochs=epochs, imgsz=640, batch=32, patience=15,
                device=0, project='runs', name=name, exist_ok=True, plots=True)
    best = str(train_out.save_dir / 'weights' / 'best.pt')
    metrics = YOLO(best).val(data=data_yaml, split='test', device=0)
    print(f'\n=== {name}: test mAP50={metrics.box.map50:.4f}  mAP50-95={metrics.box.map:.4f} ===\n')
    return best, metrics.box.map50

results = {}
results['8class'] = train_and_eval('dataset/data.yaml', 'road_damage_8class_gpu')
results['5class'] = train_and_eval('dataset_dropped5/data.yaml', 'road_damage_5class_gpu')
results['2class'] = train_and_eval('dataset_2class/data.yaml', 'road_damage_2class_gpu')

In [ ]:
print('Test mAP50 by variant:')
for name, (path, map50) in results.items():
    flag = '  <-- best' if map50 == max(m for _, m in results.values()) else ''
    print(f'  {name:8s} {map50:.4f}{flag}')

from google.colab import files
for name, (path, map50) in results.items():
    out = f'best_{name}.pt'
    shutil.copy(path, out)
    files.download(out)

## After downloading

You'll get three files: `best_8class.pt`, `best_5class.pt`, `best_2class.pt`. Tell Claude the
printed mAP50 for each and drop whichever one you want to ship into your local `Day29/` folder
as `best.pt` - it'll re-run `coding_practice/03_evaluate.py` / `04_inference.py` locally against
it, update `app.py`'s class list if needed, and update the README with the final GPU results.